# Assignment 5: Functions
##### You will build a small data audit program for a fictional apiary. The source records contain duplicate rows, inconsistent types, missing values, and invalid observations. Your program must clean the records and explain its decisions. You can use pandas unless explicitly told otherwise. Most questions require functions so that your logic can be tested independently.

In [1]:
# Starter Data

RAW_INSPECTIONS = [

{"hive_id": "H-01", "inspection_date": "2026-08-01", "temp_f": 84,

"weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"},

{"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",

"weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},

{"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",

"weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},

{"hive_id": "h-02", "inspection_date": "2026-08-02", "temp_f": "85F",

"weight_lb": 155.2, "mites": 2, "queen_seen": "yes", "notes": None},

{"hive_id": "H-02", "inspection_date": "08/16/2026", "temp_f": 91,

"weight_lb": " 151.4 ", "mites": 5, "queen_seen": True, "notes": ""},

{"hive_id": "H-02", "inspection_date": "2026-08-32", "temp_f": 92,

"weight_lb": 149.8, "mites": 6, "queen_seen": False, "notes": "bad date"},

{"hive_id": "H-03", "inspection_date": "2026-08-03", "temp_f": None,

"weight_lb": 130.0, "mites": 0, "queen_seen": True, "notes": "cool morning"},

{"hive_id": "H-03", "inspection_date": "2026-08-17", "temp_f": 96,

"weight_lb": None, "mites": 9, "queen_seen": "no", "notes": "light hive"},

{"hive_id": "H-04", "inspection_date": "2026-08-04", "temp_f": 79,

"weight_lb": 160.0, "mites": -1, "queen_seen": True, "notes": "counter reset"},

{"hive_id": "H-04", "inspection_date": "2026-08-18", "temp_f": 81,

"weight_lb": 158.5, "mites": 4, "queen_seen": True, "notes": "calm"},

{"hive_id": " H-05 ", "inspection_date": "2026-08-05", "temp_f": "72",

"weight_lb": "bad", "mites": 1, "queen_seen": True, "notes": "scale failed"},

{"hive_id": "H-05", "inspection_date": "2026-08-19", "temp_f": 74,

"weight_lb": 144.2, "mites": 3, "queen_seen": True, "notes": "retest"},

{"hive_id": "H-06", "inspection_date": "2026-08-06", "temp_f": 101,

"weight_lb": 120.0, "mites": 12, "queen_seen": False, "notes": "agitated"},

{"hive_id": "H-06", "inspection_date": "2026-08-20", "temp_f": 97,

"weight_lb": 109.4, "mites": 14, "queen_seen": False, "notes": "low stores"},

]

##### 1. (8 pts) Data Contract and Pseudocode

Before writing executable code, define what a clean inspection record means. First, write structured pseudocode for normalize_record(record). 

Use these rules:

Normalize hive_id by stripping whitespace and converting it to uppercase. Accept only H- followed by two digits.

Accept dates in YYYY-MM-DD or MM/DD/YYYY format and store the result as a datetime.date. Reject impossible dates.

Convert temp_f to float after removing a final F. None is allowed. Reject values outside -20 through 130.

Convert weight_lb to float. If conversion fails, store None rather than rejecting the record. Reject negative weight.

Convert mites to int. Reject negative values.

Convert yes and no strings for queen_seen to booleans. Reject any other value that is not already a boolean.

Convert missing notes to an empty string.

In process_log.md, include your pseudocode, two edge cases you expect to be difficult, and a prediction of how many records will be rejected before you run the program. Do not revise the prediction after running the code.

Finally, convert this to actual code. 

##### 1. Answer

normalize_record(record):
* hive_id = strip and uppercase record hive_id
    if hive_id is not "H-" + 2 digits: reject

* date = record inspection_date
    if date matches YYYY-MM-DD or MM/DD/YYYY: convert to date
    else: reject
    if date is impossible: reject

* temp = record temp_f
    if temp is None: keep None
    else:
        remove a final "F", convert to float
        if temp < -20 or temp > 130: reject
* weight = try convert record weight_lb to float
    if conversion fails: weight = None
    if weight is negative: reject

* mites = convert record mites to int
    if mites < 0: reject

* queen = record queen_seen
    if queen is "yes": queen = True
    if queen is "no": queen = False
    if queen is not True or False: reject

* notes = record notes
    if notes is None: notes = ""

In [2]:
# Normalize Records
from datetime import datetime

def normalize_record(record):
    """
    Clean and normalize raw inspection records.

    Args:
        record (dict): A raw inspection record.

    Returns:
        dict: A normalized inspection record.

    Raises:
        ValueError: If any of the fields are invalid.
    """
    
    # Hive_id: strip, uppercase, must be "H-" + 2 digits
    hive_id = str(record.get("hive_id", "")).strip().upper()
    if len(hive_id) != 4 or not hive_id.startswith("H-") or not hive_id[2:].isdigit():
        raise ValueError(f"Invalid hive_id: {hive_id}")

    # inspection_date: YYYY-MM-DD or MM/DD/YYYY, must be a real date
    raw_date = record.get("inspection_date", "")
    try:
        if "-" in raw_date:
            inspection_date = datetime.strptime(raw_date, "%Y-%m-%d").date()
        else:
            inspection_date = datetime.strptime(raw_date, "%m/%d/%Y").date()
    except ValueError as exc:
        raise ValueError(f"Invalid inspection_date: {raw_date}") from exc

    # temp_f: remove a final "F", convert to float; None allowed; -20 to 130
    raw_temp = record.get("temp_f", None)
    if raw_temp is None:
        temp_f = None
    else:
        if isinstance(raw_temp, str) and raw_temp.endswith("F"):
            raw_temp = raw_temp[:-1]
        try:
            temp_f = float(raw_temp)
        except ValueError as exc:
            raise ValueError(f"Invalid temp_f: {raw_temp}") from exc
        if temp_f < -20 or temp_f > 130:
            raise ValueError(f"temp_f out of range: {temp_f}")

    # weight_lb: convert to float; failure stores None; negative rejected
    raw_weight = record.get("weight_lb", None)
    if raw_weight is None:
        weight_lb = None
    else:
        try:
            weight_lb = float(raw_weight)
        except ValueError:
            weight_lb = None
        if weight_lb is not None and weight_lb < 0:
            raise ValueError(f"weight_lb cannot be negative: {weight_lb}")

    # mites: convert to int; negative rejected
    try:
        mites = int(record.get("mites"))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Invalid mites: {record.get('mites')}") from exc
    if mites < 0:
        raise ValueError(f"mites cannot be negative: {mites}")

    # queen_seen: "yes"/"no" to booleans; anything else non-boolean rejected
    queen_seen = record.get("queen_seen")
    if queen_seen == "yes":
        queen_seen = True
    elif queen_seen == "no":
        queen_seen = False
    if not isinstance(queen_seen, bool):
        raise ValueError(f"Invalid queen_seen: {queen_seen}")

    # notes: missing becomes an empty string
    notes = record.get("notes")
    if notes is None:
        notes = ""


    return {
        "hive_id": hive_id,
        "inspection_date": inspection_date,
        "temp_f": temp_f,
        "weight_lb": weight_lb,
        "mites": mites,
        "queen_seen": queen_seen,
        "notes": notes
    }

In [3]:
# Test above function
clean_records = []
rejected_records = []

for record in RAW_INSPECTIONS:
    try:
        clean_record = normalize_record(record)
        clean_records.append(clean_record)
    except ValueError as exc:
        rejected_records.append((record, str(exc)))

print("Clean records:")
for clean_record in clean_records:
    print(clean_record)

print("\nRejected records:")
for rejected_record, reason in rejected_records:
    print(f"{rejected_record} rejected because {reason}")

Clean records:
{'hive_id': 'H-01', 'inspection_date': datetime.date(2026, 8, 1), 'temp_f': 84.0, 'weight_lb': 142.5, 'mites': 3, 'queen_seen': True, 'notes': 'calm'}
{'hive_id': 'H-01', 'inspection_date': datetime.date(2026, 8, 15), 'temp_f': 89.0, 'weight_lb': 137.0, 'mites': 8, 'queen_seen': False, 'notes': 'queen not found'}
{'hive_id': 'H-01', 'inspection_date': datetime.date(2026, 8, 15), 'temp_f': 89.0, 'weight_lb': 137.0, 'mites': 8, 'queen_seen': False, 'notes': 'queen not found'}
{'hive_id': 'H-02', 'inspection_date': datetime.date(2026, 8, 2), 'temp_f': 85.0, 'weight_lb': 155.2, 'mites': 2, 'queen_seen': True, 'notes': ''}
{'hive_id': 'H-02', 'inspection_date': datetime.date(2026, 8, 16), 'temp_f': 91.0, 'weight_lb': 151.4, 'mites': 5, 'queen_seen': True, 'notes': ''}
{'hive_id': 'H-03', 'inspection_date': datetime.date(2026, 8, 3), 'temp_f': None, 'weight_lb': 130.0, 'mites': 0, 'queen_seen': True, 'notes': 'cool morning'}
{'hive_id': 'H-03', 'inspection_date': datetime.date

##### 2. (8 pts) (12 pts) Normalize One Record
Implement the function below without pandas.

def normalize_record(record: dict) -> tuple[dict | None, str]:

Return a new clean dictionary and an empty reason when the record is valid. Return None and a short rejection reason when it is invalid. Do not mutate the original dictionary. Create helper functions for at least date parsing and boolean conversion.

Required evidence:

A demonstration that the input dictionary remains unchanged.

A short explanation of why bad weight remains in the dataset as None while an impossible date causes rejection.


In [4]:
from datetime import datetime

# Helper functions for date parsing and boolean conversion
def parse_date(raw_date):
    """Parse YYYY-MM-DD or MM/DD/YYYY into a date; raise ValueError if invalid."""
    try:
        if "-" in raw_date:
            return datetime.strptime(raw_date, "%Y-%m-%d").date()
        return datetime.strptime(raw_date, "%m/%d/%Y").date()
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Invalid inspection_date: {raw_date}") from exc

def to_bool(value):
    """Convert "yes"/"no" to True/False; keep booleans; raise ValueError otherwise."""
    if value == "yes":
        return True
    if value == "no":
        return False
    if isinstance(value, bool):
        return value
    raise ValueError(f"Invalid queen_seen: {value}")

# Define a function to normalize inspection records
def normalize_record(record: dict) -> tuple[dict | None, str]:
    """Return (clean_record, "") if valid, or (None, reason) if rejected."""
    try:
        hive_id = str(record.get("hive_id", "")).strip().upper()
        if len(hive_id) != 4 or not hive_id.startswith("H-") or not hive_id[2:].isdigit():
            raise ValueError(f"Invalid hive_id: {hive_id}")

        inspection_date = parse_date(record.get("inspection_date"))   # helper: date parsing

        temp_f = record.get("temp_f")
        if temp_f is not None:
            temp_f = float(str(temp_f).removesuffix("F"))
            if temp_f < -20 or temp_f > 130:
                raise ValueError(f"temp_f out of range: {temp_f}")

        try:
            weight_lb = float(record.get("weight_lb"))
        except (TypeError, ValueError):
            weight_lb = None
        if weight_lb is not None and weight_lb < 0:
            raise ValueError(f"weight_lb cannot be negative: {weight_lb}")

        mites = int(record.get("mites"))
        if mites < 0:
            raise ValueError(f"mites cannot be negative: {mites}")

        queen_seen = to_bool(record.get("queen_seen"))                 # helper: boolean conversion

        notes = record.get("notes")
        if notes is None:
            notes = ""

    except (TypeError, ValueError) as exc:
        return None, str(exc)                                          # invalid: None + reason

    clean = {                                                          # a NEW dict, original untouched
        "hive_id": hive_id,
        "inspection_date": inspection_date,
        "temp_f": temp_f,
        "weight_lb": weight_lb,
        "mites": mites,
        "queen_seen": queen_seen,
        "notes": notes,
    }
    return clean, ""                                                   # valid: clean dict + empty reason

In [5]:
import copy

record = RAW_INSPECTIONS[3]         # "h-02", "85F", "yes", notes None: all values that get cleaned
before = copy.deepcopy(record)      # exact copy taken before the call

normalize_record(record)

print("Input unchanged:", record == before)

Input unchanged: True


##### 2. Why bad weight becomes None but an impossible date is rejected:
A missing or unreadable weight only loses one measurement; the rest of the
inspection (hive, date, mites, queen) is still valid and useful. An impossible
date means we can't place the inspection in time at all, so the record can't
be trusted or ordered, and it is rejected.

##### 3. (10 pts) Clean the Collection and Keep an Audit Trail
Implement:

def clean_records(records: list[dict]) -> tuple[list[dict], list[dict]]:

Call normalize_record for every source record. The second returned list must contain dictionaries with source_index, hive_id when available, and reason. After normalization, remove duplicates using the combined hive_id and inspection_date. Keep the first occurrence and record later duplicates in the rejected list with reason duplicate inspection.

Write audit_counts(rejected) that returns a dictionary mapping each rejection reason to its count. Add tests for an empty input, two duplicates, and a collection in which every row is rejected. Explain why duplicate removal happens after normalization.

In [6]:
# Define function for cleaning records
def clean_records(records: list[dict]) -> tuple[list[dict], list[dict]]:
    """Normalize every record, then remove duplicates by hive_id + inspection_date."""
    clean = []
    rejected = []
    seen = set()

    for index, record in enumerate(records):
        result, reason = normalize_record(record)
        if result is None:
            rejected.append({"source_index": index, "hive_id": record.get("hive_id"), "reason": reason})
            continue

        key = (result["hive_id"], result["inspection_date"])
        if key in seen:
            rejected.append({"source_index": index, "hive_id": result["hive_id"],
                             "reason": "duplicate inspection"})
            continue

        seen.add(key)
        clean.append(result)

    return clean, rejected


In [7]:
# Define function for audit counts
def audit_counts(rejected: list[dict]) -> dict:
    """Count how many records were rejected for each reason."""
    counts = {}
    for item in rejected:
        reason = item["reason"]
        counts[reason] = counts.get(reason, 0) + 1
    return counts


In [8]:
# Test 1: empty input
clean, rejected = clean_records([])
assert clean == [] and rejected == []
assert audit_counts(rejected) == {}
print("Empty input:", clean, rejected, audit_counts(rejected))

# Test 2: two duplicates (records 1 and 2 are the same H-01 inspection)
clean, rejected = clean_records([RAW_INSPECTIONS[1], RAW_INSPECTIONS[2]])
assert len(clean) == 1
assert audit_counts(rejected) == {"duplicate inspection": 1}
print("Two duplicates:", len(clean), "kept |", rejected)

# Test 3: every row rejected (impossible date + negative mites)
clean, rejected = clean_records([RAW_INSPECTIONS[5], RAW_INSPECTIONS[8]])
assert clean == []
assert len(rejected) == 2
print("All rejected:", audit_counts(rejected))


Empty input: [] [] {}
Two duplicates: 1 kept | [{'source_index': 1, 'hive_id': 'H-01', 'reason': 'duplicate inspection'}]
All rejected: {'Invalid inspection_date: 2026-08-32': 1, 'mites cannot be negative: -1': 1}


##### 3. Why duplicate removal happens after normalization:
The same inspection can be written differently in the raw data, such as "h-02" vs
"H-02", or "08/16/2026" vs "2026-08-16". Comparing raw values would treat these
as different inspections and keep both. Normalizing first puts every record in
the same format, so matching hive_id + inspection_date reliably finds true duplicates.
